# 02 -- Window-Size Ablation: TP53 Mutation Subtype CNN (11/21/51/101bp)

Trains the **same** `SimpleCNN` (from `src/model.py`), with the **same**
hyperparameters, on the hotspot and rare-variant datasets at three additional
context window sizes: 11bp, 51bp, 101bp. 21bp is **not** retrained here --
those results already exist in `results/main/` from `01_main_experiment.ipynb`
and are pulled in for comparison only.

All reusable logic (architecture, training loop, metrics, baselines) comes
from `src/data.py`, `src/model.py`, `src/train.py`, `src/metrics.py` --
exactly the modules `01_main_experiment.ipynb` already uses. Nothing is
reimplemented here.

**Before any training happens**, this notebook re-verifies (rather than
assumes) that the test-set `position_id` set is identical across all four
window sizes for each dataset -- this was a known bug in an earlier version
of the pipeline (splitting on raw `position_id` instead of sequence-identity
clusters leaked duplicate sequence content across train/test). If that check
fails, the notebook stops before spending time on 6 training runs.


In [1]:
SEED = 42
import random, numpy as np, torch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [2]:
import os
import sys
import json

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data import CLASSES, load_split, load_position_table, MutationDataset, position_majority_ceiling
from src.model import SimpleCNN, count_trainable_params
from src.train import compute_class_weights, train_model, predict
from src.metrics import compute_all_metrics, random_baseline, majority_baseline

ABLATION_WINDOW_SIZES = [11, 51, 101]   # trained in this notebook
EXISTING_WINDOW_SIZE = 21               # pulled in from results/main/, not retrained
ALL_WINDOW_SIZES = sorted(ABLATION_WINDOW_SIZES + [EXISTING_WINDOW_SIZE])
DATASETS = ['hotspot', 'rare']

SPLITS_ROOT = os.path.join(PROJECT_ROOT, 'data', 'splits')
MAIN_RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'main')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'window_ablation')
PRED_DIR = os.path.join(RESULTS_DIR, 'predictions')
CKPT_DIR = os.path.join(RESULTS_DIR, 'checkpoints')

for d in (RESULTS_DIR, PRED_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

# position_id -> cluster_id map, used for cluster-resampled accuracy CIs below.
# cluster_id is window-size independent (a property of position_id alone, see
# notebooks/data_pipeline.ipynb Stage 5), so the same map applies at every
# window size tested in this notebook.
POSITION_TABLE_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'position_table.csv')
position_to_cluster = load_position_table(POSITION_TABLE_PATH).set_index('position_id')['cluster_id']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Project root:          {PROJECT_ROOT}")
print(f"Splits root:           {SPLITS_ROOT}")
print(f"Existing (21bp) results:{MAIN_RESULTS_DIR}")
print(f"Ablation results dir:  {RESULTS_DIR}")
print(f"Window sizes trained here: {ABLATION_WINDOW_SIZES}")
print(f"Window size reused from notebook 01: {EXISTING_WINDOW_SIZE}")
print(f"Device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == 'cuda' else ""))


Project root:          C:\Users\danya\Documents\projects\tp53_mutation_subtype
Splits root:           C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\splits
Existing (21bp) results:C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\main
Ablation results dir:  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\window_ablation
Window sizes trained here: [11, 51, 101]
Window size reused from notebook 01: 21
Device: cuda (NVIDIA GeForce RTX 5090)


## Critical verification (run before any training): test-set `position_id`
sets must be identical across all four window sizes

This is the actual guarantee the whole ablation depends on: if window sizes
disagreed on which positions are in the test set, accuracy differences
across window sizes could just reflect different (easier/harder) test sets,
not the effect of window size itself. Checked here for `train`, `val`, and
`test` -- and for `Sequence` overlap across splits too, since that's the
leak this pipeline was previously fixed for.


In [3]:
print("Position-id consistency check across all four window sizes")
print("=" * 70)

split_position_ids = {}  # split_position_ids[window][dataset][split] = set(position_id)
split_sequences = {}     # same shape, but Sequence strings, for the leak check

for w in ALL_WINDOW_SIZES:
    split_position_ids[w] = {}
    split_sequences[w] = {}
    for name in DATASETS:
        split_position_ids[w][name] = {}
        split_sequences[w][name] = {}
        for split_name in ('train', 'val', 'test'):
            path = os.path.join(SPLITS_ROOT, f'window_{w}', name, f'{split_name}.csv')
            df = load_split(path)
            split_position_ids[w][name][split_name] = set(df['position_id'])
            split_sequences[w][name][split_name] = set(df['Sequence'])

all_ok = True
for name in DATASETS:
    for split_name in ('train', 'val', 'test'):
        id_sets = {w: split_position_ids[w][name][split_name] for w in ALL_WINDOW_SIZES}
        reference = id_sets[ALL_WINDOW_SIZES[0]]
        matches = all(id_sets[w] == reference for w in ALL_WINDOW_SIZES[1:])
        all_ok = all_ok and matches
        sizes = {w: len(id_sets[w]) for w in ALL_WINDOW_SIZES}
        print(f"{name}/{split_name}: sizes per window = {sizes}  "
              f"identical across all {len(ALL_WINDOW_SIZES)} window sizes: {matches}")

    # No Sequence may cross a train/val/test boundary, at any window size.
    for w in ALL_WINDOW_SIZES:
        seqs = split_sequences[w][name]
        leak_tv = seqs['train'] & seqs['val']
        leak_tt = seqs['train'] & seqs['test']
        leak_vt = seqs['val'] & seqs['test']
        no_leak = not (leak_tv or leak_tt or leak_vt)
        all_ok = all_ok and no_leak
        print(f"{name}/window={w}: no Sequence overlap between any split pair: {no_leak}")
    print()

print("=" * 70)
if all_ok:
    print("OVERALL: PASS -- position sets agree across all window sizes, "
          "and no sequence leak at any window size. Safe to proceed with training.")
else:
    raise AssertionError(
        "Position-id / sequence-leak consistency check FAILED -- STOPPING before training. "
        "This indicates a mismatch between window-size datasets that would invalidate "
        "any accuracy-vs-window-size comparison. Do not proceed until this is fixed "
        "in notebooks/data_pipeline.ipynb."
    )


Position-id consistency check across all four window sizes


hotspot/train: sizes per window = {11: 5304, 21: 5304, 51: 5304, 101: 5304}  identical across all 4 window sizes: True
hotspot/val: sizes per window = {11: 1143, 21: 1143, 51: 1143, 101: 1143}  identical across all 4 window sizes: True
hotspot/test: sizes per window = {11: 1136, 21: 1136, 51: 1136, 101: 1136}  identical across all 4 window sizes: True
hotspot/window=11: no Sequence overlap between any split pair: True
hotspot/window=21: no Sequence overlap between any split pair: True
hotspot/window=51: no Sequence overlap between any split pair: True
hotspot/window=101: no Sequence overlap between any split pair: True

rare/train: sizes per window = {11: 4041, 21: 4041, 51: 4041, 101: 4041}  identical across all 4 window sizes: True
rare/val: sizes per window = {11: 915, 21: 915, 51: 915, 101: 915}  identical across all 4 window sizes: True
rare/test: sizes per window = {11: 884, 21: 884, 51: 884, 101: 884}  identical across all 4 window sizes: True
rare/window=11: no Sequence overlap

## Model architecture (reused, unchanged, from `src/model.py`)

`SimpleCNN` uses global max pooling over the sequence dimension and has no
hardcoded `seq_len` anywhere in `forward()` -- confirmed here by running a
forward pass at all three ablation window sizes, not assumed.


In [4]:
_sanity_model = SimpleCNN()
n_params = count_trainable_params(_sanity_model)
print(_sanity_model)
print(f"\nTrainable parameters: {n_params:,}")
assert n_params == 7206, f"Expected 7,206 trainable parameters, got {n_params:,}"

for w in ABLATION_WINDOW_SIZES:
    x = torch.randn(2, 4, w)
    out = _sanity_model(x)
    assert out.shape == (2, 6), f"window={w}: unexpected output shape {out.shape}"
    print(f"window={w:>3}bp: forward pass OK, output shape {tuple(out.shape)}")

del _sanity_model


SimpleCNN(
  (conv1): Conv1d(4, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=64, out_features=6, bias=True)
  (relu): ReLU(inplace=True)
)

Trainable parameters: 7,206
window= 11bp: forward pass OK, output shape (2, 6)
window= 51bp: forward pass OK, output shape (2, 6)
window=101bp: forward pass OK, output shape (2, 6)


## Train: 6 models from scratch -- {11, 51, 101}bp x {hotspot, rare}

Identical config to `01_main_experiment.ipynb`: Adam lr=1e-4, batch_size=128,
max_epochs=50, cross-entropy with inverse-frequency class weights, early
stopping on validation accuracy (patience=15), ReduceLROnPlateau on
validation loss, gradient clipping max_norm=1.0, FP16 mixed precision on
CUDA -- via the same `train_model()` function, not a reimplementation.


In [5]:
splits = {}       # splits[window][dataset][split] -> DataFrame
models = {}        # models[(window, dataset)] -> trained model
histories = {}      # histories[(window, dataset)] -> history dict

for w in ABLATION_WINDOW_SIZES:
    splits[w] = {}
    for name in DATASETS:
        splits[w][name] = {
            split_name: load_split(os.path.join(SPLITS_ROOT, f'window_{w}', name, f'{split_name}.csv'))
            for split_name in ('train', 'val', 'test')
        }

        print("=" * 70)
        print(f"TRAINING: window={w}bp, dataset={name}")
        print("=" * 70)

        torch.manual_seed(SEED)  # re-seed so each of the 6 models is trained independently and reproducibly

        train_ds = MutationDataset(splits[w][name]['train'])
        val_ds = MutationDataset(splits[w][name]['val'])

        class_weights = compute_class_weights(train_ds.y.numpy(), device=device)
        print(f"Class weights ({CLASSES}): {class_weights.cpu().numpy().round(3)}")

        model = SimpleCNN()
        model, history = train_model(
            model, train_ds, val_ds, device,
            max_epochs=50, batch_size=128, lr=1e-4,
            patience=15, lr_patience=5, lr_factor=0.5,
            grad_clip_norm=1.0, class_weights=class_weights,
            seed=SEED, verbose=True,
        )

        checkpoint_path = os.path.join(CKPT_DIR, f'window_{w}_{name}.pt')
        torch.save({
            'model_state_dict': model.state_dict(),
            'seed': SEED,
            'dataset': name,
            'window_size': w,
            'best_val_accuracy': history['best_val_accuracy'],
            'best_epoch': history['best_epoch'],
            'stopped_epoch': history['stopped_epoch'],
            'classes': CLASSES,
        }, checkpoint_path)

        print(f"\nBest val_accuracy: {history['best_val_accuracy']:.4f} "
              f"(epoch {history['best_epoch']}, stopped at {history['stopped_epoch']})")
        print(f"Checkpoint saved -> {checkpoint_path}\n")

        models[(w, name)] = model
        histories[(w, name)] = history


TRAINING: window=11bp, dataset=hotspot


Class weights (['C>A', 'C>G', 'C>T', 'T>A', 'T>C', 'T>G']): [0.885 1.958 0.347 2.811 1.147 3.97 ]


epoch   1/50  train_loss=1.0351  val_loss=1.1846  val_acc=0.3220  lr=1.00e-04 *


epoch   2/50  train_loss=0.8180  val_loss=1.1464  val_acc=0.3452  lr=1.00e-04 *


epoch   3/50  train_loss=0.7841  val_loss=1.0612  val_acc=0.3583  lr=1.00e-04 *


...

epoch  12/50  train_loss=1.5765  val_loss=1.7957  val_acc=0.2678  lr=5.00e-05


epoch  13/50  train_loss=1.5848  val_loss=1.8006  val_acc=0.2686  lr=5.00e-05


epoch  14/50  train_loss=1.5618  val_loss=1.8031  val_acc=0.2671  lr=5.00e-05


epoch  15/50  train_loss=1.5508  val_loss=1.8057  val_acc=0.2671  lr=5.00e-05


epoch  16/50  train_loss=1.5630  val_loss=1.8029  val_acc=0.2686  lr=2.50e-05
Early stopping at epoch 16 (no val_accuracy improvement for 15 epochs)

Best val_accuracy: 0.2973 (epoch 1, stopped at 16)
Checkpoint saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\window_ablation\checkpoints\window_101_rare.pt



## Evaluate each of the 6 models: metrics, baselines, position ceiling

Same metric functions as notebook 01 (`compute_all_metrics`, bootstrap CI,
etc.). Baselines here are random and majority only -- CpG rule and the
trinucleotide logistic regression aren't part of this ablation (they were
already established as baselines in notebook 01, and majority-class /
random are the two that are directly comparable window-to-window).

The majority-class baseline should be **identical** to the corresponding
21bp value in `results/main/summary.csv`, since the majority class is a
property of the training label distribution, not of window size (the
positions and their labels are the same at every window size -- only the
`Sequence` context around them differs). This is checked explicitly below.


In [6]:
ablation_metrics = {}       # ablation_metrics[(window, dataset)] -> dict
ablation_pred_frames = {}   # ablation_pred_frames[(window, dataset)] -> DataFrame

for w in ABLATION_WINDOW_SIZES:
    for name in DATASETS:
        train_df = splits[w][name]['train']
        test_df = splits[w][name]['test']
        test_ds = MutationDataset(test_df)
        y_test = test_ds.y.numpy()

        cnn_preds, cnn_probs = predict(models[(w, name)], test_ds, device)

        rand_preds = random_baseline(len(y_test), seed=SEED)
        y_train = MutationDataset(train_df).y.numpy()
        maj_preds, majority_class = majority_baseline(y_train, len(y_test))

        cluster_ids_test = position_to_cluster.loc[test_df['position_id']].values
        metrics = compute_all_metrics(y_test, cnn_preds, cluster_ids=cluster_ids_test)
        metrics['baselines'] = {
            'random_accuracy': float((rand_preds == y_test).mean()),
            'majority_accuracy': float((maj_preds == y_test).mean()),
            'majority_class': majority_class,
        }
        metrics['position_majority_ceiling'] = position_majority_ceiling(test_df)
        metrics['n_test_instances'] = int(len(y_test))
        metrics['n_test_positions'] = int(test_df['position_id'].nunique())
        metrics['window_size'] = w
        metrics['dataset'] = name
        metrics['best_epoch'] = histories[(w, name)]['best_epoch']
        metrics['stopped_epoch'] = histories[(w, name)]['stopped_epoch']

        ablation_metrics[(w, name)] = metrics

        pred_df = pd.DataFrame({
            'position_id': test_df['position_id'].values,
            'window_size': w,
            'dataset': name,
            'true_label': [CLASSES[i] for i in y_test],
            'predicted_label': [CLASSES[i] for i in cnn_preds],
            'probabilities': list(cnn_probs),
        })[['position_id', 'window_size', 'dataset', 'true_label', 'predicted_label', 'probabilities']]

        pred_path = os.path.join(PRED_DIR, f'window_{w}_{name}.parquet')
        pred_df.to_parquet(pred_path, engine='pyarrow', index=False)
        ablation_pred_frames[(w, name)] = pred_df

        print(f"window={w:>3}bp  {name:<7}  accuracy={metrics['accuracy']:.4f}  "
              f"mcc={metrics['mcc']:.4f}  majority_acc={metrics['baselines']['majority_accuracy']:.4f}  "
              f"-> {pred_path}")


window= 11bp  hotspot  accuracy=0.2878  mcc=0.2089  majority_acc=0.6494  -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\window_ablation\predictions\window_11_hotspot.parquet


window= 11bp  rare     accuracy=0.3929  mcc=0.2415  majority_acc=0.3582  -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\window_ablation\predictions\window_11_rare.parquet


window= 51bp  hotspot  accuracy=0.1821  mcc=0.1171  majority_acc=0.6494  -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\window_ablation\predictions\window_51_hotspot.parquet


window= 51bp  rare     accuracy=0.1448  mcc=-0.0071  majority_acc=0.3582  -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\window_ablation\predictions\window_51_rare.parquet


window=101bp  hotspot  accuracy=0.1434  mcc=0.0710  majority_acc=0.6494  -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\window_ablation\predictions\window_101_hotspot.parquet


window=101bp  rare     accuracy=0.2273  mcc=0.0611  majority_acc=0.3582  -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\window_ablation\predictions\window_101_rare.parquet


## Cross-check: majority-class baseline must match notebook 01's 21bp value

If it doesn't, the position set (or its labels) differs between window
sizes for that dataset -- a real problem, flagged explicitly rather than
silently reported, per the same policy notebook 01 used for its published-
numbers sanity check.


In [7]:
main_summary = pd.read_csv(os.path.join(MAIN_RESULTS_DIR, 'summary.csv'))
main_summary_by_dataset = main_summary.set_index('dataset')

MAJORITY_MATCH_TOLERANCE = 1e-9

majority_mismatches = []
for name in DATASETS:
    existing_majority = main_summary_by_dataset.loc[name, 'majority_accuracy']
    for w in ABLATION_WINDOW_SIZES:
        current_majority = ablation_metrics[(w, name)]['baselines']['majority_accuracy']
        diff = abs(current_majority - existing_majority)
        match = diff < MAJORITY_MATCH_TOLERANCE
        print(f"{name}/window={w:>3}bp: majority_accuracy={current_majority:.6f}  "
              f"(21bp reference={existing_majority:.6f})  match: {match}")
        if not match:
            majority_mismatches.append((name, w, existing_majority, current_majority, diff))

print()
if majority_mismatches:
    print(f"FLAGGED: {len(majority_mismatches)} window/dataset combo(s) have a majority-class "
          "baseline that disagrees with the 21bp reference -- this points to a position-set "
          "mismatch between window sizes and needs investigating before trusting the ablation:")
    for name, w, existing, current, diff in majority_mismatches:
        print(f"  {name}/window={w}: 21bp={existing:.6f}  {w}bp={current:.6f}  diff={diff:.6f}")
else:
    print("OK: majority-class baseline is identical to the 21bp reference for every "
          "window size and dataset, as expected (majority class depends on the label "
          "distribution, not the window size).")


hotspot/window= 11bp: majority_accuracy=0.649385  (21bp reference=0.649385)  match: True
hotspot/window= 51bp: majority_accuracy=0.649385  (21bp reference=0.649385)  match: True
hotspot/window=101bp: majority_accuracy=0.649385  (21bp reference=0.649385)  match: True
rare/window= 11bp: majority_accuracy=0.358171  (21bp reference=0.358171)  match: True
rare/window= 51bp: majority_accuracy=0.358171  (21bp reference=0.358171)  match: True
rare/window=101bp: majority_accuracy=0.358171  (21bp reference=0.358171)  match: True

OK: majority-class baseline is identical to the 21bp reference for every window size and dataset, as expected (majority class depends on the label distribution, not the window size).


## Assemble `results/window_ablation/summary.csv` -- 8 rows (4 window sizes x 2 datasets)

The 6 rows trained here, plus the 2 existing 21bp rows read in from
`results/main/summary.csv` (not recomputed).


In [8]:
SUMMARY_COLUMNS = [
    'window_size', 'dataset', 'accuracy', 'accuracy_ci_low', 'accuracy_ci_high',
    'accuracy_ci_low_clustered', 'accuracy_ci_high_clustered',
    'balanced_accuracy', 'macro_f1', 'weighted_f1', 'mcc',
    'majority_accuracy', 'random_accuracy', 'position_majority_ceiling',
]

summary_rows = []

# 6 newly trained rows
for w in ABLATION_WINDOW_SIZES:
    for name in DATASETS:
        m = ablation_metrics[(w, name)]
        summary_rows.append({
            'window_size': w,
            'dataset': name,
            'accuracy': m['accuracy'],
            'accuracy_ci_low': m['accuracy_ci_low'],
            'accuracy_ci_high': m['accuracy_ci_high'],
            'accuracy_ci_low_clustered': m['accuracy_ci_low_clustered'],
            'accuracy_ci_high_clustered': m['accuracy_ci_high_clustered'],
            'balanced_accuracy': m['balanced_accuracy'],
            'macro_f1': m['macro_f1'],
            'weighted_f1': m['weighted_f1'],
            'mcc': m['mcc'],
            'majority_accuracy': m['baselines']['majority_accuracy'],
            'random_accuracy': m['baselines']['random_accuracy'],
            'position_majority_ceiling': m['position_majority_ceiling'],
        })

# 2 existing 21bp rows, pulled in from results/main/summary.csv -- not retrained
for name in DATASETS:
    row = main_summary_by_dataset.loc[name]
    summary_rows.append({
        'window_size': EXISTING_WINDOW_SIZE,
        'dataset': name,
        'accuracy': row['accuracy'],
        'accuracy_ci_low': row['accuracy_ci_low'],
        'accuracy_ci_high': row['accuracy_ci_high'],
        'accuracy_ci_low_clustered': row['accuracy_ci_low_clustered'],
        'accuracy_ci_high_clustered': row['accuracy_ci_high_clustered'],
        'balanced_accuracy': row['balanced_accuracy'],
        'macro_f1': row['macro_f1'],
        'weighted_f1': row['weighted_f1'],
        'mcc': row['mcc'],
        'majority_accuracy': row['majority_accuracy'],
        'random_accuracy': row['random_accuracy'],
        'position_majority_ceiling': row['position_majority_ceiling'],
    })

summary_df = pd.DataFrame(summary_rows, columns=SUMMARY_COLUMNS)
summary_df = summary_df.sort_values(['dataset', 'window_size']).reset_index(drop=True)

summary_path = os.path.join(RESULTS_DIR, 'summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f"Summary written -> {summary_path} ({len(summary_df)} rows)\n")
summary_df


Summary written -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\window_ablation\summary.csv (8 rows)



,window_size,dataset,accuracy,accuracy_ci_low,accuracy_ci_high,accuracy_ci_low_clustered,accuracy_ci_high_clustered,balanced_accuracy,macro_f1,weighted_f1,mcc,majority_accuracy,random_accuracy,position_majority_ceiling
0,11,hotspot,0.287812,0.285162,0.290267,0.149159,0.581146,0.414195,0.304407,0.315156,0.208870,0.649385,0.165744,0.803228
1,21,hotspot,0.132200,0.130272,0.134130,0.053702,0.238118,0.259782,0.155013,0.093986,0.066582,0.649385,0.165744,0.803228
2,51,hotspot,0.182089,0.179839,0.184364,0.094403,0.317616,0.330019,0.218559,0.154692,0.117070,0.649385,0.165744,0.803228
3,101,hotspot,0.143444,0.141434,0.145481,0.064578,0.260416,0.258205,0.183652,0.130517,0.070990,0.649385,0.165744,0.803228
4,11,rare,0.392887,0.372566,0.411939,0.300705,0.493733,0.341460,0.337700,0.401608,0.241469,0.358171,0.161727,0.726926
5,21,rare,0.218036,0.201937,0.235817,0.151688,0.292324,0.198719,0.194176,0.221522,0.026101,0.358171,0.161727,0.726926
6,51,rare,0.144793,0.130398,0.158764,0.080688,0.214610,0.167258,0.142167,0.140903,-0.007121,0.358171,0.161727,0.726926
7,101,rare,0.227350,0.209992,0.243861,0.140858,0.322844,0.217422,0.192914,0.219179,0.061141,0.358171,0.161727,0.726926


## Hamming-distance analysis: how close is each held-out test sequence to its nearest training sequence?

For each window size and dataset, computed over the **unique** Sequence values
in `data/splits/window_{w}/{name}/{train,test}.csv` (not raw instance rows,
since many instances share an identical Sequence): for every unique test
sequence, the Hamming distance to every unique training sequence is computed,
and the minimum is kept (nearest-neighbour distance). `exact_overlap` counts
test sequences with a nearest-neighbour distance of exactly 0, which should
always be 0 given the leakage check in `data_pipeline.ipynb` guarantees no
exact-duplicate Sequence crosses a train/test boundary at any window size.
This addresses whether the 11bp window's outlying accuracy (Window Ablation
Results) reflects held-out test sequences sitting unusually close to training
sequences at that window size.

In [9]:
def min_hamming_stats(train_seqs, test_seqs):
    train_arr = np.array([list(s) for s in train_seqs])
    test_arr = np.array([list(s) for s in test_seqs])
    min_dists = np.empty(len(test_arr))
    for i, t in enumerate(test_arr):
        min_dists[i] = (train_arr != t).sum(axis=1).min()
    return min_dists


hamming_rows = []
for w in ALL_WINDOW_SIZES:
    for name in DATASETS:
        train_df = pd.read_csv(os.path.join(SPLITS_ROOT, f'window_{w}', name, 'train.csv'))
        test_df = pd.read_csv(os.path.join(SPLITS_ROOT, f'window_{w}', name, 'test.csv'))
        train_seqs = train_df['Sequence'].unique()
        test_seqs = test_df['Sequence'].unique()

        min_dists = min_hamming_stats(train_seqs, test_seqs)
        hamming_rows.append({
            'window_size': w,
            'dataset': name,
            'train_unique_seqs': len(train_seqs),
            'test_unique_seqs': len(test_seqs),
            'exact_overlap': int((min_dists == 0).sum()),
            'mean_min_hamming': float(min_dists.mean()),
            'median_min_hamming': float(np.median(min_dists)),
            'mean_hamming_pct_of_length': float(min_dists.mean() / w * 100),
            'frac_dist_le_1': float((min_dists <= 1).mean()),
            'frac_dist_le_2': float((min_dists <= 2).mean()),
        })

hamming_df = pd.DataFrame(hamming_rows).sort_values(['dataset', 'window_size']).reset_index(drop=True)
hamming_path = os.path.join(RESULTS_DIR, 'hamming_distance.csv')
hamming_df.to_csv(hamming_path, index=False)
print(f"Saved -> {hamming_path}\n")
pd.set_option('display.width', 160)
print(hamming_df.to_string(index=False))


Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\window_ablation\hamming_distance.csv

 window_size dataset  train_unique_seqs  test_unique_seqs  exact_overlap  mean_min_hamming  median_min_hamming  mean_hamming_pct_of_length  frac_dist_le_1  frac_dist_le_2
          11 hotspot                322                70              0          3.214286                 3.0                   29.220779        0.014286        0.114286
          21 hotspot                331                72              0          9.013889                 9.0                   42.923280        0.000000        0.000000
          51 hotspot                359                74              0         27.135135                27.5                   53.206147        0.000000        0.000000
         101 hotspot                402                90              0         59.433333                60.0                   58.844884        0.000000        0.000000
          11    rare              

## Figure: accuracy vs. window size

Descriptive only -- two lines (hotspot, rare), shaded region from the
bootstrap 95% CI. No trend test (Cochran-Armitage or otherwise) is fitted or
reported here, by design.


In [10]:
fig, ax = plt.subplots(figsize=(7, 5))
colors = {'hotspot': 'tab:blue', 'rare': 'tab:orange'}

for name in DATASETS:
    sub = summary_df[summary_df['dataset'] == name].sort_values('window_size')
    ax.plot(sub['window_size'], sub['accuracy'], marker='o', label=name, color=colors[name])
    ax.fill_between(sub['window_size'], sub['accuracy_ci_low_clustered'], sub['accuracy_ci_high_clustered'],
                     color=colors[name], alpha=0.2)

ax.set_xticks(ALL_WINDOW_SIZES)
ax.set_xlabel('Window size (bp)')
ax.set_ylabel('Accuracy')
ax.set_title('CNN test accuracy vs. context window size')
ax.legend()
ax.grid(True, alpha=0.3)

fig_path = os.path.join(RESULTS_DIR, 'figure_accuracy_vs_window.png')
fig.tight_layout()
fig.savefig(fig_path, dpi=200)
plt.close(fig)
print(f"Figure saved -> {fig_path}")


Figure saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\window_ablation\figure_accuracy_vs_window.png


## Final summary: all 8 rows, and the plain-language accuracy-vs-window trend

In [11]:
print("=" * 100)
print("FINAL RESULTS -- 02_window_ablation (11 / 21 / 51 / 101 bp)")
print("=" * 100)
pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)
print(summary_df.to_string(index=False))

print("\n" + "=" * 100)
print("Accuracy vs. window size -- plain description (not a fitted trend statistic)")
print("=" * 100)
for name in DATASETS:
    sub = summary_df[summary_df['dataset'] == name].sort_values('window_size')
    accs = sub['accuracy'].tolist()
    windows = sub['window_size'].tolist()

    diffs = [accs[i + 1] - accs[i] for i in range(len(accs) - 1)]
    if all(d > 0 for d in diffs):
        direction = 'trends UP monotonically as window size increases'
    elif all(d < 0 for d in diffs):
        direction = 'trends DOWN monotonically as window size increases'
    elif max(accs) - min(accs) < 0.01:
        direction = 'is essentially FLAT across window sizes (range < 1 percentage point)'
    else:
        direction = 'is NON-MONOTONIC across window sizes (no consistent up/down trend)'

    print(f"\n{name}:")
    for w, a in zip(windows, accs):
        print(f"  window={w:>3}bp: accuracy={a:.4f}")
    print(f"  -> {name} accuracy {direction}")
    print(f"     (range: {min(accs):.4f} to {max(accs):.4f}, "
          f"delta={max(accs) - min(accs):.4f})")


FINAL RESULTS -- 02_window_ablation (11 / 21 / 51 / 101 bp)
 window_size dataset  accuracy  accuracy_ci_low  accuracy_ci_high  accuracy_ci_low_clustered  accuracy_ci_high_clustered  balanced_accuracy  macro_f1  weighted_f1       mcc  majority_accuracy  random_accuracy  position_majority_ceiling
          11 hotspot  0.287812         0.285162          0.290267                   0.149159                    0.581146           0.414195  0.304407     0.315156  0.208870           0.649385         0.165744                   0.803228
          21 hotspot  0.132200         0.130272          0.134130                   0.053702                    0.238118           0.259782  0.155013     0.093986  0.066582           0.649385         0.165744                   0.803228
          51 hotspot  0.182089         0.179839          0.184364                   0.094403                    0.317616           0.330019  0.218559     0.154692  0.117070           0.649385         0.165744                   0.803